In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import string

nltk.download("punkt")

# Load QA dataset
df = pd.read_excel("../data/noticiasQA.xlsx")

# Load DeepSeek (or any generative LLM)
model_name = "deepseek-ai/deepseek-llm-7b-base"  # You can use mistralai/Mistral-7B-Instruct-v0.2, etc.

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Normalize function for EM and F1
def normalize(text):
    text = text.lower().translate(str.maketrans("", "", string.punctuation))
    return word_tokenize(text)

def f1_score(prediction, ground_truth):
    pred_tokens = normalize(prediction)
    truth_tokens = normalize(ground_truth)
    common = Counter(pred_tokens) & Counter(truth_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)
    if num_same == 0:
        return 0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(truth_tokens)
    return 2 * (precision * recall) / (precision + recall)

def exact_match(prediction, ground_truth):
    return int(normalize(prediction) == normalize(ground_truth))

# Evaluation
em_total = 0
f1_total = 0
n = len(df)

# Inference loop
for _, row in tqdm(df.iterrows(), total=n):
    context = row["context"]
    question = row["question"]
    ground_truth = str(row["answer"])

    # Build Spanish QA prompt
    prompt = f"""Contesta la siguiente pregunta usando el contexto provisto.

Contexto: {context}

Pregunta: {question}

Respuesta:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.7,
        top_p=0.95
    )
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_answer = output_text.split("Respuesta:")[-1].strip()

    em_total += exact_match(generated_answer, ground_truth)
    f1_total += f1_score(generated_answer, ground_truth)

# Results
print(f"\nExact Match: {em_total / n * 100:.2f}%")
print(f"F1 Score: {f1_total / n * 100:.2f}%")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\guill\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


tokenizer_config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

C:\Users\guill\Documents\GitHub\nlp-media-framing\.venv\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guill\.cache\huggingface\hub\models--deepseek-ai--deepseek-llm-7b-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.json:   0%|          | 0.00/4.61M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/584 [00:00<?, ?B/s]

...

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

  0%|          | 0/25 [00:00<?, ?it/s]C:\Users\guill\Documents\GitHub\nlp-media-framing\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
C:\Users\guill\Documents\GitHub\nlp-media-framing\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
100%|██████████| 25/25 [58:10<00:00, 139.63s/it]


Exact Match: 4.00%
F1 Score: 30.45%
